###  Transactions Data Chunking

This notebook splits the raw transactions dataset into:
- an initial batch (historical load)
- an incremental batch (newer data)

The chunked data will be used for downstream Bronze ingestion.
The notebook is designed to be idempotent and safe for multiple runs.


In [0]:
# Read raw transactions data from  volume
# This reads all CSV files under the transactions folder

df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/transactions/")
)


In [0]:
from pyspark.sql.functions import to_timestamp

# Convert created_at column to timestamp for safe date comparisons
df = df.withColumn(
    "created_at",
    to_timestamp("created_at")
)


In [0]:
# Just a sanity check
from pyspark.sql.functions import min, max

# Validate time range after conversion
df.select(
    min("created_at").alias("min_created_at"),
    max("created_at").alias("max_created_at")
).show()


In [0]:
# Cutoff date for batch vs incremental split
cutoff_date = "2024-10-31"


In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("created_date", to_date("created_at"))

cutoff_date = "2024-10-31"

df_batch = df.filter(df.created_date <= cutoff_date)
df_incremental = df.filter(df.created_date > cutoff_date)


In [0]:
# Validate chunking logic

total_count = df.count()
batch_count = df_batch.count()
incremental_count = df_incremental.count()

print("Total records:", total_count)
print("Batch records:", batch_count)
print("Incremental records:", incremental_count)
print("Batch + Incremental:", batch_count + incremental_count)


In [0]:
# Write initial batch chunk to  volume
# Overwrite mode ensures idempotent behavior
(
    df_batch
    .write
    .mode("overwrite")   # safe for multiple runs
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/chunked_transactions/batch/")
)


In [0]:

# write incremental batch chunk to volume
(
    df_incremental
    .write
    .mode("overwrite")  
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/chunked_transactions/incremental/")
)


In [0]:
# date range validations after chunking for batch
from pyspark.sql.functions import min, max

batch_path = "/Volumes/workspace/default/coffee_raw_volume/chunked_transactions/batch"

df_batch = (
    spark.read
    .option("header", True)
    .csv(batch_path)
)

df_batch.select(
    min("created_at").alias("min_created_at"),
    max("created_at").alias("max_created_at")
).show(truncate=False)


In [0]:
# date range validations after chunking for incremental
incremental_path = "/Volumes/workspace/default/coffee_raw_volume/chunked_transactions/incremental"

df_incremental = (
    spark.read
    .option("header", True)
    .csv(incremental_path)
)

df_incremental.select(
    min("created_at").alias("min_created_at"),
    max("created_at").alias("max_created_at")
).show(truncate=False)
